In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *
from textblob import TextBlob

#pd.set_option("display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata

In [ ]:
df_lead_paragraph = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/HourSentiment.pkl")

In [ ]:
df_lead_paragraph

In [ ]:
df_lead_paragraph["Date"] = df_lead_paragraph.index
df_lead_paragraph=df_lead_paragraph.sort_values(by="Date")

In [ ]:
# New words and values
new_words = {
    'crushes': 100,
    'beats': 5,
    'misses': -5,
    'trouble': -10,
    'falls': -100,
    "dividends":100,
    "dividend":100,
    "earning":100,
    "ipo":100,
    "alliance":100,
    "enhance":100,
    "enhanc":100
}
# Instantiate the sentiment intensity analyzer with the existing lexicon
vader = SentimentIntensityAnalyzer()
# Update the lexicon
vader.lexicon.update(new_words)

In [ ]:
pos_word_list=[]
neu_word_list=[]
neg_word_list=[]
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['Sentiment']] = ''
df[["lead_paragraph"]] = ''
df[["polarity"]] = ''
df[["subjectivity"]] = ''
df[["polarityB"]] = ''
# df[['GI_Negative']] = ''
# df[['GI_Positive']] = ''
# df[['GI_Sentiment']] = ''
# df[["GI_Polarity"]] = ''
# df[["GI_Subjectivity"]] = ''

In [ ]:
df_lead_paragraph.dropna(inplace=True)

In [ ]:
df_lead_paragraph = df_lead_paragraph.loc[df_lead_paragraph["NewsCount"]!=0]

In [ ]:
debug=False
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0
import datetime 
import pysentiment2 as ps
hiv4 = ps.HIV4()

rows_list = []
for date, row in df_lead_paragraph.T.iteritems():
    Hour = int(row["Hour"])
    print(Hour)
    sentence = unicodedata.normalize('NFKD', row['lead_paragraph'])#.encode('ascii','ignore').decode('ascii')
    # sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")

    prc = row['NewsCount']
    ss = vader.polarity_scores(sentence)

    Neg = ss['neg']
    Neu = ss['neu']
    Pos = ss['pos']
    Compound = ss['compound']
    Sentiment=0
    prc = row['NewsCount']
    d = datetime.timedelta(days = 100)
    formater = "%Y-%m-%d"
    d1 = datetime.datetime.strptime(date,formater)
    date_100 = d1 - d
    date_100=str(date_100)
    mean_prc = df_lead_paragraph.loc[date_100:date]['NewsCount'].mean()
    NewsCount_factor = (prc/mean_prc)
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            Sentiment = (100*(ss['neg'] / (ss['pos'] + ss['neg'])))*NewsCount_factor

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']
    
    ############## TEXTBLOB ##############
    blob = TextBlob(sentence)
    print(date)
    Polarity = blob.sentiment.polarity
    Subjectivity = blob.sentiment.subjectivity
    polarityB = ""
    if (blob.sentiment.polarity > 0 ):
        polarityB = "Positive"
    elif (blob.sentiment.polarity < 0 ):
        polarityB = "Negative"
    elif (blob.sentiment.polarity == 0 ):
        polarityB = "Neutral"

    new_row = {"Date":date, "Hour": Hour, "compound":Compound, "neg":Neg, "neu":Neu, "pos":Pos, "sentiment":Sentiment, "polarity": Polarity, "subjectivity":Subjectivity, "polarityB":polarityB,"lead_paragraph":sentence,"NewsCount":row["NewsCount"]}
    rows_list.append(new_row)
df = pd.DataFrame(data=rows_list, columns=["Date", "Hour", "compound", "neg", "neu", "pos", "sentiment", "polarity", "subjectivity", "polarityB","lead_paragraph", "NewsCount"])
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/SentimentHourly.pkl")

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

In [ ]:
del df